# CPAM Disk Anomaly Detection v8
## Two-Stage Pipeline: Isolation Forest + Gaussian Noise Injection + XGBoost
### Changes from V4:
- Removed `scale_pos_weight` from XGBoost.
- Integrated Gaussian Noise Injection to physically balance the training data (50/50).
- Maintained 2% IF pseudo-contamination for realistic anomaly thresholds.
- **V8 Update**: Dropped IF scores from XGBoost feature matrix to fix label leakage, forcing XGBoost to learn true physical root causes.

## 0. Setup & Imports

In [ ]:
# Install required packages (run once on Colab)
!pip install xgboost optuna shap --quiet imbalanced-learn

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    silhouette_score,
    f1_score,
    roc_curve,
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
import optuna
import shap

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Global plot style
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 100,
})

print("All libraries loaded successfully.")

## 1. Load & Inspect Data

In [ ]:
# Upload pipeline_disk_metrics.csv to Colab or update the path below
DATA_PATH = 'pipeline_disk_metrics.csv'

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

# Count total lines to report how many are skipped
with open(DATA_PATH, 'r') as _f:
    total_lines = sum(1 for _ in _f) - 1  # subtract header

# Some rows have extra fields (19 instead of 18) due to pipeline tags.
# These are safely skipped — they represent < 0.1% of data.
df_raw = pd.read_csv(DATA_PATH, on_bad_lines='skip')
skipped = total_lines - len(df_raw)
if skipped > 0:
    print(f"Skipped {skipped} malformed row(s) during CSV parsing.")

df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])
df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)

print(f"Shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}): {df_raw.columns.tolist()}")
print(f"\nTime range: {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")
print(f"Duration:   {df_raw['timestamp'].max() - df_raw['timestamp'].min()}")

missing = df_raw.isnull().sum()
if missing.sum() == 0:
    print("\nMissing values: None (dataset is clean)")
else:
    print(f"\nMissing values:\n{missing[missing > 0]}")

display(df_raw.head())
display(df_raw.describe())


## 2. Exploratory Data Analysis (EDA)

### 2.1 Time-Series Overview

In [ ]:
ts_metrics = [
    ('used_gb',              'Disk Used (GB)',            '#2C3E50'),
    ('util_pct',             'Disk Utilization (%)',      '#E74C3C'),
    ('free_gb',              'Disk Free (GB)',            '#27AE60'),
    ('write_bytes_sec',      'Write Bytes/sec',           '#8E44AD'),
    ('read_bytes_sec',       'Read Bytes/sec',            '#F39C12'),
    ('write_iops',           'Write IOPS',                '#E67E22'),
    ('avg_write_latency_ms', 'Avg Write Latency (ms)',    '#16A085'),
    ('avg_read_latency_ms',  'Avg Read Latency (ms)',     '#C0392B'),
]

fig, axes = plt.subplots(4, 2, figsize=(18, 18), sharex=True)
fig.suptitle('Time-Series Overview of Disk Metrics', fontsize=16, y=1.01)

for idx, (col, title, color) in enumerate(ts_metrics):
    ax = axes[idx // 2, idx % 2]
    ax.plot(df_raw['timestamp'], df_raw[col], color=color, linewidth=0.4, alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(col, fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("Key Observations:")
print(f"  - used_gb increases monotonically from {df_raw['used_gb'].min():.2f} to {df_raw['used_gb'].max():.2f} GB")
print(f"  - util_pct ranges from {df_raw['util_pct'].min():.1f}% to {df_raw['util_pct'].max():.1f}%")
print(f"  - Many read I/O values are zero (write-heavy workload)")

### 2.2 Distribution Analysis

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
# Exclude timestamp-derived and constant columns
plot_cols = [c for c in numeric_cols if c not in ['total_gb']]

n_cols = 3
n_rows = (len(plot_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
fig.suptitle('Feature Distributions (Histograms)', fontsize=16, y=1.01)

for idx, col in enumerate(plot_cols):
    ax = axes[idx // n_cols, idx % n_cols]
    data = df_raw[col].dropna()
    ax.hist(data, bins=80, color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(col, fontsize=10)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1, label=f'mean={data.mean():.2f}')
    ax.axvline(data.median(), color='green', linestyle='--', linewidth=1, label=f'median={data.median():.2f}')
    ax.legend(fontsize=7)

# Hide unused subplots
for idx in range(len(plot_cols), n_rows * n_cols):
    axes[idx // n_cols, idx % n_cols].set_visible(False)

plt.tight_layout()
plt.show()

# Zero-value analysis
print("\nZero-Value Analysis (% of rows with value = 0):")
print("=" * 50)
for col in numeric_cols:
    zero_pct = (df_raw[col] == 0).mean() * 100
    if zero_pct > 1:
        print(f"  {col:30s}: {zero_pct:6.1f}%")

### 2.3 Correlation Heatmap

In [ ]:
corr_cols = [
    'util_pct', 'used_gb', 'free_gb',
    'write_bytes_sec', 'read_bytes_sec',
    'write_iops', 'read_iops',
    'avg_write_latency_ms', 'avg_read_latency_ms',
    'kafka_size_gb', 'opensearch_size_gb',
]

corr_matrix = df_raw[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHighly correlated pairs (|r| > 0.9):")
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            print(f"  {corr_matrix.index[i]} <-> {corr_matrix.columns[j]}: r = {r:.3f}")

### 2.4 Service-Specific Disk Usage (Kafka, OpenSearch, Logstash)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Service-Specific Disk Usage Over Time', fontsize=16, y=1.03)

services = [
    ('kafka_size_gb',      'Kafka',      '#E74C3C'),
    ('opensearch_size_gb', 'OpenSearch',  '#3498DB'),
    ('logstash_size_gb',   'Logstash',    '#2ECC71'),
]

for idx, (col, name, color) in enumerate(services):
    ax = axes[idx]
    ax.plot(df_raw['timestamp'], df_raw[col], color=color, linewidth=1)
    ax.set_title(f'{name} Disk Usage (GB)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel('GB')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Service summary
print("\n" + "=" * 65)
print("SERVICE-SPECIFIC DISK USAGE SUMMARY")
print("=" * 65)
for col, name, _ in services:
    s = df_raw[col]
    print(f"\n{name}:")
    print(f"  Range:     {s.min():.4f} GB -> {s.max():.4f} GB  (delta = {s.max() - s.min():.4f} GB)")
    print(f"  Mean/Std:  {s.mean():.4f} / {s.std():.4f} GB")
    nonzero_pct = (s > 0).mean() * 100
    print(f"  Non-zero:  {nonzero_pct:.1f}% of rows")
    if s.max() == 0 and s.min() == 0:
        print(f"  >> No {name} activity detected in this dataset (all zeros).")
    elif s.max() - s.min() < 0.01:
        print(f"  >> Negligible {name} growth detected.")

## 3. Data Cleaning & Feature Engineering

### Key Design Decisions
- **Drop redundant columns**: `write_mb_sec`/`read_mb_sec` (identical to bytes_sec / 1e6), `total_gb` (constant), `node_id`/`os` (single-valued)
- **Rate-of-change features**: Since `used_gb` and `kafka_size_gb` are monotonically increasing, raw values are not useful for anomaly detection. Instead, we engineer *rates of change* (via `diff()`) which capture **how fast** things are changing.
- **Rolling Z-scores**: Capture how unusual a value is relative to recent history (5-minute rolling window)
- **Log transforms**: Applied to heavy-tailed I/O metrics via `log1p()` to reduce skew

In [ ]:
WINDOW = 60  # Rolling window size: 60 samples ~ 5 minutes at 5-sec intervals

df_feat = pd.DataFrame()

# ── 1. Rate-of-change features (diff) ──
df_feat['disk_growth_rate']      = df_raw['used_gb'].diff().fillna(0)
df_feat['kafka_growth_rate']     = df_raw['kafka_size_gb'].diff().fillna(0)
df_feat['opensearch_growth_rate']= df_raw['opensearch_size_gb'].diff().fillna(0)
df_feat['util_pct_change']       = df_raw['util_pct'].diff().fillna(0)

# ── 2. Log-transformed I/O features ──
# log1p handles zeros gracefully: log1p(0) = 0
for col in ['write_bytes_sec', 'read_bytes_sec', 'write_iops', 'read_iops']:
    df_feat[f'log_{col}'] = np.log1p(df_raw[col].values)

# ── 3. Latency & health features (kept as-is) ──
df_feat['avg_write_latency_ms'] = df_raw['avg_write_latency_ms'].values
df_feat['avg_read_latency_ms']  = df_raw['avg_read_latency_ms'].values
df_feat['util_pct']             = df_raw['util_pct'].values

# ── 4. Rolling window features (Z-scores and volatility) ──
rolling_targets = ['log_write_bytes_sec', 'log_write_iops', 'disk_growth_rate', 'kafka_growth_rate']

for col in rolling_targets:
    src = df_feat[col]
    rmean = src.rolling(WINDOW, min_periods=WINDOW).mean()
    rstd  = src.rolling(WINDOW, min_periods=WINDOW).std()
    df_feat[f'rstd_{col}']   = rstd
    df_feat[f'zscore_{col}'] = np.where(rstd > 1e-10, (src - rmean) / rstd, 0.0)

# ── 5. Derived ratio features ──
# Log-difference captures write/read ratio in log-space (numerically stable)
df_feat['io_write_read_ratio'] = df_feat['log_write_bytes_sec'] - df_feat['log_read_bytes_sec']
# Remaining disk capacity as a fraction
df_feat['remaining_disk_pct']  = 1.0 - df_feat['util_pct'] / 100.0

# ── 6. Drop initial rows with NaN from rolling windows ──
DROP_ROWS = WINDOW
df_feat = df_feat.iloc[DROP_ROWS:].reset_index(drop=True)

# Keep aligned timestamps and raw data for plotting later
timestamps = df_raw['timestamp'].iloc[DROP_ROWS:].reset_index(drop=True)
df_plot_raw = df_raw.iloc[DROP_ROWS:].reset_index(drop=True)

# Verify no NaN remain
nan_counts = df_feat.isnull().sum()
assert nan_counts.sum() == 0, f"Unexpected NaN values:\n{nan_counts[nan_counts > 0]}"

print(f"Feature matrix shape: {df_feat.shape}")
print(f"\nEngineered features ({len(df_feat.columns)}):")
for i, col in enumerate(df_feat.columns, 1):
    print(f"  {i:2d}. {col}")

display(df_feat.head())
display(df_feat.describe())

### 3.1 Engineered Feature Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr = df_feat.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.3,
            ax=ax, annot_kws={'fontsize': 6})
ax.set_title('Engineered Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Stage 1 — Isolation Forest with Optuna Tuning

We train **three** specialized Isolation Forest models, each focusing on a different aspect of disk behavior:

| IF Model | Focus | Features |
|----------|-------|----------|
| **Throughput** | I/O volume anomalies | log(write/read bytes/sec, IOPS), Z-scores |
| **Latency** | Latency/health anomalies | latencies, utilization, util change |
| **Capacity** | Disk growth anomalies | growth rates, Z-scores, remaining capacity |

Each model's hyperparameters are tuned using **Optuna** with **silhouette score** as the objective (measures how well-separated the anomaly/normal groups are).

In [ ]:
# -- Define feature groups for each Isolation Forest --

if_feature_groups = {
    'throughput': [
        'log_write_bytes_sec', 'log_read_bytes_sec',
        'log_write_iops', 'log_read_iops',
        'zscore_log_write_bytes_sec', 'zscore_log_write_iops',
    ],
    'latency': [
        'avg_write_latency_ms', 'avg_read_latency_ms',
        'util_pct', 'util_pct_change',
    ],
    'capacity': [
        'disk_growth_rate', 'kafka_growth_rate', 'opensearch_growth_rate',
        'zscore_disk_growth_rate', 'zscore_kafka_growth_rate',
        'remaining_disk_pct',
    ],
}

scalers = {}
scaled_groups = {}

for group_name, cols in if_feature_groups.items():
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_feat[cols].values)
    scalers[group_name] = scaler
    scaled_groups[group_name] = X_scaled
    print(f"IF group '{group_name}': {len(cols)} features, shape {X_scaled.shape}")

print("\nFeature groups defined and scaled.")


In [ ]:
# -- Isolation Forest Hyperparameter Tuning with Optuna --

def if_objective(trial, X_scaled):
    """Optuna objective for IF tuning. Maximizes silhouette score."""
    n_estimators  = trial.suggest_int('n_estimators', 100, 500)
    max_samples   = trial.suggest_float('max_samples', 0.5, 1.0)
    max_features  = trial.suggest_float('max_features', 0.5, 1.0)
    contamination = trial.suggest_float('contamination', 0.02, 0.15)

    model = IsolationForest(
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        contamination=contamination,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_scaled)
    labels = model.predict(X_scaled)
    binary_labels = (labels == -1).astype(int)

    n_anom = binary_labels.sum()
    n_norm = len(binary_labels) - n_anom
    if n_anom < 5 or n_norm < 5:
        return -1.0  # Degenerate split

    # Subsample for silhouette score (O(n^2) otherwise)
    n_sample = min(3000, len(X_scaled))
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(len(X_scaled), n_sample, replace=False)

    try:
        score = silhouette_score(X_scaled[idx], binary_labels[idx],
                                random_state=RANDOM_STATE)
    except Exception:
        return -1.0

    return score

# Run Optuna for each IF group
N_IF_TRIALS = 30
if_studies = {}
if_best_params = {}

for group_name in if_feature_groups.keys():
    print(f"\n{'=' * 60}")
    print(f"Tuning Isolation Forest: {group_name.upper()}")
    print(f"Features: {if_feature_groups[group_name]}")
    print(f"{'=' * 60}")

    X_g = scaled_groups[group_name]

    study = optuna.create_study(direction='maximize',
                               study_name=f'IF_{group_name}')
    study.optimize(
        lambda trial, _X=X_g: if_objective(trial, _X),
        n_trials=N_IF_TRIALS,
        show_progress_bar=True,
    )

    if_studies[group_name] = study
    if_best_params[group_name] = study.best_params

    print(f"Best silhouette score: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")


In [ ]:
# -- Fit tuned Isolation Forests and generate anomaly scores --

if_models = {}
if_score_cols = []

for group_name, best_params in if_best_params.items():
    print(f"\nFitting tuned IF: {group_name} ...")

    model = IsolationForest(
        n_estimators=best_params['n_estimators'],
        max_samples=best_params['max_samples'],
        max_features=best_params['max_features'],
        contamination=best_params['contamination'],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    X_g = scaled_groups[group_name]
    model.fit(X_g)
    if_models[group_name] = model

    # Negate decision_function so higher = more anomalous
    score = -model.decision_function(X_g)
    col_name = f'if_{group_name}_score'
    df_feat[col_name] = score
    if_score_cols.append(col_name)

    n_anom = (model.predict(X_g) == -1).sum()
    print(f"  Anomalies detected: {n_anom} ({n_anom / len(X_g) * 100:.1f}%)")
    print(f"  Score range: [{score.min():.4f}, {score.max():.4f}]")

print(f"\nAll IF models fitted. Added columns: {if_score_cols}")
print(f"Updated feature matrix shape: {df_feat.shape}")


In [ ]:
# ── Visualize IF scores over time ──

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)
fig.suptitle('Isolation Forest Anomaly Scores Over Time', fontsize=16, y=1.01)

colors = {'throughput': '#E74C3C', 'latency': '#3498DB', 'capacity': '#2ECC71'}

for idx, group_name in enumerate(if_feature_groups.keys()):
    ax = axes[idx]
    col = f'if_{group_name}_score'
    scores = df_feat[col]
    color = colors[group_name]

    ax.plot(timestamps, scores, color=color, linewidth=0.4, alpha=0.7)
    ax.set_title(f'IF {group_name.title()} Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Anomaly Score')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

    # Mark threshold at the tuned contamination quantile
    contam = if_best_params[group_name]['contamination']
    threshold = np.quantile(scores, 1 - contam)
    ax.axhline(y=threshold, color='red', linestyle=':', alpha=0.7,
               label=f'Threshold (contam={contam:.2f})')
    ax.legend(fontsize=9, loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
axes[-1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Score distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('IF Score Distributions', fontsize=14, y=1.03)

for idx, group_name in enumerate(if_feature_groups.keys()):
    col = f'if_{group_name}_score'
    ax = axes[idx]
    ax.hist(df_feat[col], bins=100, color=colors[group_name], alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{group_name.title()} Scores', fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# ── Isolation Forest Threshold Analysis ──
# Analyze how different percentage cutoffs affect the anomaly boundary

# 1. Calculate the average IF score across all three forests
avg_if_scores = (df_feat['if_throughput_score'] + 
                 df_feat['if_latency_score'] + 
                 df_feat['if_capacity_score']) / 3

# 2. Sort the scores
sorted_scores = np.sort(avg_if_scores)
n_points = len(sorted_scores)

# 3. Calculate threshold cutoffs
idx_10_pct = int(n_points * 0.90) # Top 10%
idx_05_pct = int(n_points * 0.95) # Top 5%
idx_02_pct = int(n_points * 0.98) # Top 2%

val_10_pct = sorted_scores[idx_10_pct]
val_05_pct = sorted_scores[idx_05_pct]
val_02_pct = sorted_scores[idx_02_pct]

# 4. Plot the sorted curve with threshold lines
fig, ax = plt.subplots(figsize=(16, 8))

# Plot the curve
ax.plot(range(n_points), sorted_scores, color='#34495E', linewidth=2, label='Sorted Average IF Score')

# Highlight the top 2% (current setting) in red
ax.scatter(range(idx_02_pct, n_points), sorted_scores[idx_02_pct:], 
           color='#E74C3C', s=10, zorder=5, label='Top 2% Anomalies')

# Add threshold lines
ax.axvline(x=idx_10_pct, color='#F39C12', linestyle='--', linewidth=1.5, 
           label=f'10% Threshold (Score: {val_10_pct:.3f})')
ax.axvline(x=idx_05_pct, color='#3498DB', linestyle='--', linewidth=1.5, 
           label=f'5% Threshold (Score: {val_05_pct:.3f})')
ax.axvline(x=idx_02_pct, color='#E74C3C', linestyle='-', linewidth=2, 
           label=f'2% Threshold (Score: {val_02_pct:.3f})')

ax.set_title('Average Isolation Forest Score vs Contamination Thresholds', fontsize=16, fontweight='bold')
ax.set_xlabel('Data Points (Sorted by Average IF Score)', fontsize=12)
ax.set_ylabel('Average IF Anomaly Score', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("THRESHOLD ANALYSIS SUMMARY")
print("=" * 60)
print(f"If we set contamination to 10%: {n_points - idx_10_pct} points flagged as anomalies.")
print(f"If we set contamination to 5%:  {n_points - idx_05_pct} points flagged as anomalies.")
print(f"If we set contamination to 2%:  {n_points - idx_02_pct} points flagged as anomalies.")
print("-" * 60)
print("Look at the curve: If the curve is flat and suddenly spikes upward, ")
print("the threshold should be placed exactly where the spike begins.")


## 5. Pseudo-Labels & Temporal Train/Test Split

### Pseudo-Label Generation
Since no ground-truth labels exist, we create pseudo-labels from the IF scores:
- Compute a **composite anomaly score** = weighted average of the three IF scores
- Label the top N% (by composite score) as anomalies

### Temporal Split
**Critical**: We use a temporal split (first 75% train, last 25% test) instead of random splitting.  
Random splitting would cause data leakage since adjacent time points are correlated.

In [ ]:
# ── Composite anomaly score (weighted average of IF scores) ──

# Weight each IF score equally (can be adjusted based on domain knowledge)
composite_score = (
    df_feat['if_throughput_score'] +
    df_feat['if_latency_score'] +
    df_feat['if_capacity_score']
) / 3.0

# Also consider maximum score (a point is anomalous if ANY IF flags it)
max_score = np.maximum.reduce([
    df_feat['if_throughput_score'],
    df_feat['if_latency_score'],
    df_feat['if_capacity_score'],
])

# Use maximum score for pseudo-labeling (more conservative)
PSEUDO_CONTAMINATION = 0.02  # Top 2% are pseudo-anomalies (realistic for production)
threshold = np.quantile(max_score, 1 - PSEUDO_CONTAMINATION)
pseudo_labels = (max_score >= threshold).astype(int)

df_feat['composite_score'] = composite_score
df_feat['max_if_score'] = max_score

print(f"Composite score threshold (p={1 - PSEUDO_CONTAMINATION:.0%}): {threshold:.4f}")
print(f"\nPseudo-label distribution:")
print(f"  Normal (0):  {(pseudo_labels == 0).sum():>6d}  ({(pseudo_labels == 0).mean()*100:.1f}%)")
print(f"  Anomaly (1): {(pseudo_labels == 1).sum():>6d}  ({(pseudo_labels == 1).mean()*100:.1f}%)")

In [ ]:
# -- Temporal Train/Test Split (75/25) --

xgb_feature_cols = [c for c in df_feat.columns
                    if c not in ['composite_score', 'max_if_score', 'if_throughput_score', 'if_latency_score', 'if_capacity_score']]

X_all = df_feat[xgb_feature_cols].values
y_all = pseudo_labels

# 75/25 Temporal Split — train on the past, test on the future
split_idx = int(len(X_all) * 0.75)

X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
timestamps_train = timestamps.iloc[:split_idx]
timestamps_test  = timestamps.iloc[split_idx:]

print(f"Feature columns ({len(xgb_feature_cols)}): {xgb_feature_cols}")
print(f"\nTrain set: {X_train.shape[0]} samples ({y_train.sum()} pseudo-anomalies, {y_train.mean()*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({y_test.sum()} pseudo-anomalies, {y_test.mean()*100:.1f}%)")
print(f"\nTrain period: {timestamps_train.iloc[0]} to {timestamps_train.iloc[-1]}")
print(f"Test period:  {timestamps_test.iloc[0]} to {timestamps_test.iloc[-1]}")

assert len(np.unique(y_train)) == 2, "Train set has only one class!"
assert len(np.unique(y_test)) == 2, "Test set has only one class!"


In [ ]:
# ── Gaussian Noise Injection (Synthetic Anomaly Augmentation) ──
# Instead of drawing synthetic lines between anomalies (SMOTE-style),
# we duplicate real anomalies and add tiny, physically realistic noise.
# This avoids generating synthetic points inside the normal data region.

NOISE_STD_FRACTION = 0.02  # 2% of each feature's std-dev as noise amplitude
TARGET_ANOMALY_RATIO = 0.50  # 50/50 balance

print('Class distribution BEFORE Gaussian Noise Injection:')
print(f'  Normal:  {(y_train == 0).sum()} ({np.mean(y_train == 0)*100:.1f}%)')
print(f'  Anomaly: {(y_train == 1).sum()} ({np.mean(y_train == 1)*100:.1f}%)')

# Separate anomaly rows from the training set
X_anom = X_train[y_train == 1]  # real anomaly rows (numpy array)
X_norm = X_train[y_train == 0]  # normal rows

# How many synthetic anomalies do we need to reach the target ratio?
n_normal = len(X_norm)
n_real_anom = len(X_anom)
# target_ratio = n_anom / (n_normal + n_anom)  =>  solve for n_anom
n_total_anom_needed = int((TARGET_ANOMALY_RATIO * n_normal) / (1 - TARGET_ANOMALY_RATIO))
n_synthetic_needed = max(0, n_total_anom_needed - n_real_anom)

print(f'\nTarget anomaly ratio: {TARGET_ANOMALY_RATIO*100:.0f}%')
print(f'Real anomalies available: {n_real_anom}')
print(f'Synthetic anomalies to generate: {n_synthetic_needed}')

# Generate synthetic anomalies by jittering real ones
rng = np.random.RandomState(RANDOM_STATE)

# Compute per-feature noise scale (fraction of feature std)
feature_stds = X_anom.std(axis=0)
noise_scale = NOISE_STD_FRACTION * feature_stds

# Repeatedly sample from real anomalies and add Gaussian noise
n_iters = (n_synthetic_needed // n_real_anom) + 1
synthetic_parts = []
for _ in range(n_iters):
    noise = rng.randn(*X_anom.shape) * noise_scale
    synthetic_parts.append(X_anom + noise)

X_synthetic = np.vstack(synthetic_parts)[:n_synthetic_needed]
y_synthetic = np.ones(n_synthetic_needed, dtype=int)

# Combine: original training data + synthetic anomalies
X_train_sm = np.vstack([X_train, X_synthetic])
y_train_sm = np.concatenate([y_train, y_synthetic])

# Shuffle so anomalies aren't all at the end
shuffle_idx = rng.permutation(len(X_train_sm))
X_train_sm = X_train_sm[shuffle_idx]
y_train_sm = y_train_sm[shuffle_idx]

print('\nClass distribution AFTER Gaussian Noise Injection:')
print(f'  Normal:  {(y_train_sm == 0).sum()} ({np.mean(y_train_sm == 0)*100:.1f}%)')
print(f'  Anomaly: {(y_train_sm == 1).sum()} ({np.mean(y_train_sm == 1)*100:.1f}%)')
print(f'\nGenerated {n_synthetic_needed} synthetic anomaly points via Gaussian Noise Injection.')


## 6. Stage 2 — XGBoost with Optuna Hyperparameter Tuning

This is the **core deliverable**. We use Optuna to perform Bayesian optimization over XGBoost hyperparameters.

- **Objective**: Maximize **AUC-PR** (Average Precision) — the best metric for imbalanced anomaly detection
- **Cross-Validation**: `TimeSeriesSplit` (3 folds) within the training set — respects temporal order
- **Early Stopping**: Each trial trains up to 1000 trees with early stopping at 50 rounds of no improvement
- **Search Space**: 8 hyperparameters including depth, learning rate, regularization, sampling

In [ ]:
# ── XGBoost Hyperparameter Tuning with Optuna ──

def xgb_objective(trial):
    """Optuna objective: maximize mean AUC-PR across StratifiedKFold folds."""
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    ap_scores = []

    for train_idx, val_idx in skf.split(X_train_sm, y_train_sm):
        # SMOTE returns numpy arrays in this environment, so we use standard bracket indexing
        X_tr_f = X_train_sm[train_idx]
        y_tr_f = y_train_sm[train_idx]
        X_va_f = X_train_sm[val_idx]
        y_va_f = y_train_sm[val_idx]

        # Skip if a fold has only one class
        if len(np.unique(y_tr_f)) < 2 or len(np.unique(y_va_f)) < 2:
            continue

        model = xgb.XGBClassifier(
            n_estimators=1000,
            early_stopping_rounds=50,
            **params,
            eval_metric='aucpr',
            random_state=RANDOM_STATE,
            n_jobs=-1,
            tree_method='hist',
            verbosity=0,
        )

        model.fit(
            X_tr_f, y_tr_f,
            eval_set=[(X_va_f, y_va_f)],
            verbose=False,
        )

        y_prob_f = model.predict_proba(X_va_f)[:, 1]
        ap = average_precision_score(y_va_f, y_prob_f)
        ap_scores.append(ap)

    return np.mean(ap_scores) if ap_scores else 0.0


# Run optimization
N_XGB_TRIALS = 100
print(f"\nStarting Optuna XGBoost optimization ({N_XGB_TRIALS} trials)...")
print("This may take 10-20 minutes on Colab.\n")

xgb_study = optuna.create_study(direction='maximize', study_name='XGBoost_tuning')
xgb_study.optimize(xgb_objective, n_trials=N_XGB_TRIALS, show_progress_bar=True)

print(f"\n{'=' * 60}")
print(f"OPTUNA RESULTS")
print(f"{'=' * 60}")
print(f"Best AUC-PR (CV mean): {xgb_study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for k, v in xgb_study.best_params.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.6f}")
    else:
        print(f"  {k:25s}: {v}")
print(f"{'=' * 60}")


In [ ]:
# ── Optuna Results Visualization ──

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('XGBoost Optuna Tuning Results', fontsize=16, y=1.03)

# 1. Optimization history
ax = axes[0]
trial_numbers = [t.number for t in xgb_study.trials]
trial_values  = [t.value for t in xgb_study.trials]
best_so_far   = np.maximum.accumulate(trial_values)

ax.scatter(trial_numbers, trial_values, alpha=0.4, s=20, color='steelblue', label='Trial AUC-PR')
ax.plot(trial_numbers, best_so_far, color='red', linewidth=2, label='Best so far')
ax.set_xlabel('Trial Number')
ax.set_ylabel('AUC-PR (CV Mean)')
ax.set_title('Optimization History', fontweight='bold')
ax.legend()

# 2. Hyperparameter importance
ax = axes[1]
# Compute importance by correlation of param values with trial outcomes
param_names = list(xgb_study.best_params.keys())
importances = []
for pname in param_names:
    pvals = [t.params.get(pname, np.nan) for t in xgb_study.trials]
    valid_mask = ~np.isnan(pvals) & ~np.isnan(trial_values)
    if valid_mask.sum() > 2:
        r = abs(np.corrcoef(np.array(pvals)[valid_mask],
                            np.array(trial_values)[valid_mask])[0, 1])
        importances.append(r if not np.isnan(r) else 0.0)
    else:
        importances.append(0.0)

sorted_idx = np.argsort(importances)[::-1]
ax.barh([param_names[i] for i in sorted_idx],
        [importances[i] for i in sorted_idx],
        color='steelblue', alpha=0.8)
ax.set_xlabel('|Correlation| with AUC-PR')
ax.set_title('Hyperparameter Importance (approx.)', fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Print top 5 trials
print("\nTop 5 Trials:")
print(f"{'Trial':>6s}  {'AUC-PR':>8s}  {'max_depth':>10s}  {'lr':>10s}  {'subsample':>10s}  {'colsample':>10s}")
sorted_trials = sorted(xgb_study.trials, key=lambda t: t.value, reverse=True)
for t in sorted_trials[:5]:
    p = t.params
    print(f"{t.number:6d}  {t.value:8.4f}  {p['max_depth']:10d}  {p['learning_rate']:10.5f}  {p['subsample']:10.4f}  {p['colsample_bytree']:10.4f}")

In [ ]:
# ── Train Final XGBoost with Best Hyperparameters ──

best_params = xgb_study.best_params

# Create validation split from end of training data for early stopping
val_split = int(len(X_train) * 0.85)
X_tr_final = X_train[:val_split]
y_tr_final = y_train[:val_split]
X_va_final = X_train[val_split:]
y_va_final = y_train[val_split:]

final_spw = (y_tr_final == 0).sum() / max((y_tr_final == 1).sum(), 1)

model_tuned = xgb.XGBClassifier(
    n_estimators=1000,
    early_stopping_rounds=50,
    **best_params,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    verbosity=0,
)

print("Training final tuned XGBoost model...")
model_tuned.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_va_final, y_va_final)],
    verbose=False,
)

best_iter = model_tuned.best_iteration
print(f"Best iteration (early stopping): {best_iter}")
print(f"\nTuned model trained successfully.")

## 7. Evaluation & Model Comparison

> **Note**: These metrics are evaluated against pseudo-labels (derived from IF), not ground-truth.  
> They measure how well XGBoost has learned and generalized the IF-detected patterns on unseen future data.  
> The temporal split ensures the test data comes **after** the training data, preventing data leakage.

In [ ]:
# ── Evaluation on Temporal Test Set ──

y_prob_tuned = model_tuned.predict_proba(X_test)[:, 1]
y_pred_tuned = (y_prob_tuned >= 0.5).astype(int)

auc_roc_tuned = roc_auc_score(y_test, y_prob_tuned)
auc_pr_tuned  = average_precision_score(y_test, y_prob_tuned)
f1_tuned      = f1_score(y_test, y_pred_tuned)

print("=" * 60)
print("TUNED XGBOOST — TEST SET EVALUATION")
print("=" * 60)
print(f"AUC-ROC: {auc_roc_tuned:.4f}")
print(f"AUC-PR:  {auc_pr_tuned:.4f}")
print(f"F1:      {f1_tuned:.4f}")
print(f"\nClassification Report (pseudo-labels):")
print(classification_report(y_test, y_pred_tuned, digits=3,
                            target_names=['Normal', 'Anomaly']))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomaly'])
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Tuned XGBoost — Confusion Matrix', fontweight='bold')

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_prob_tuned)
axes[1].plot(rec, prec, color='#E74C3C', linewidth=2, label=f'Tuned XGBoost (AUC-PR={auc_pr_tuned:.3f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.6, label='Random baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Compare: Default (old) vs Tuned XGBoost ──

# Train with the EXACT default params from the old notebook (v2)

model_default = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    verbosity=0,
)

print("Training DEFAULT (old) XGBoost model for comparison...")
model_default.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_va_final, y_va_final)],
    verbose=False,
)

y_prob_default = model_default.predict_proba(X_test)[:, 1]
y_pred_default = (y_prob_default >= 0.5).astype(int)

auc_roc_default = roc_auc_score(y_test, y_prob_default)
auc_pr_default  = average_precision_score(y_test, y_prob_default)
f1_default      = f1_score(y_test, y_pred_default)

# ── Comparison Table ──
print("\n" + "=" * 65)
print("MODEL COMPARISON: DEFAULT vs TUNED")
print("=" * 65)
print(f"{'Metric':<20s} {'Default (v2)':<15s} {'Tuned (v3)':<15s} {'Improvement':<15s}")
print("-" * 65)

for metric_name, v_default, v_tuned in [
    ('AUC-ROC', auc_roc_default, auc_roc_tuned),
    ('AUC-PR', auc_pr_default, auc_pr_tuned),
    ('F1 Score', f1_default, f1_tuned),
]:
    diff = v_tuned - v_default
    arrow = '+' if diff > 0 else ''
    print(f"{metric_name:<20s} {v_default:<15.4f} {v_tuned:<15.4f} {arrow}{diff:<+14.4f}")

print("=" * 65)

# ── Visual comparison ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar comparison
metrics = ['AUC-ROC', 'AUC-PR', 'F1']
default_vals = [auc_roc_default, auc_pr_default, f1_default]
tuned_vals   = [auc_roc_tuned, auc_pr_tuned, f1_tuned]

x = np.arange(len(metrics))
width = 0.35
axes[0].bar(x - width/2, default_vals, width, label='Default (v2)', color='#95A5A6', alpha=0.8)
axes[0].bar(x + width/2, tuned_vals,   width, label='Tuned (v3)',   color='#E74C3C', alpha=0.8)
axes[0].set_ylabel('Score')
axes[0].set_title('Default vs Tuned XGBoost', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.05)
for i, (d, t) in enumerate(zip(default_vals, tuned_vals)):
    axes[0].text(i - width/2, d + 0.01, f'{d:.3f}', ha='center', fontsize=9)
    axes[0].text(i + width/2, t + 0.01, f'{t:.3f}', ha='center', fontsize=9)

# Overlaid PR curves
prec_d, rec_d, _ = precision_recall_curve(y_test, y_prob_default)
prec_t, rec_t, _ = precision_recall_curve(y_test, y_prob_tuned)
axes[1].plot(rec_d, prec_d, color='#95A5A6', linewidth=2, label=f'Default (AUC-PR={auc_pr_default:.3f})')
axes[1].plot(rec_t, prec_t, color='#E74C3C', linewidth=2, label=f'Tuned (AUC-PR={auc_pr_tuned:.3f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('PR Curve Comparison', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7.1 Anomaly Timeline Visualization

This is the **most important visual validation** in an unsupervised pipeline.  
We overlay detected anomalies on key time-series metrics to verify they align with visually "interesting" events.

In [ ]:
# ── Score all data with the tuned model and plot anomalies ──

y_prob_all = model_tuned.predict_proba(X_all)[:, 1]
y_pred_all = (y_prob_all >= 0.5).astype(int)
anom_mask  = y_pred_all == 1

n_anom = anom_mask.sum()
print(f"Total anomalies detected across full dataset: {n_anom} ({n_anom/len(y_pred_all)*100:.1f}%)")
print(f"  In training period: {(y_pred_all[:split_idx] == 1).sum()}")
print(f"  In test period:     {(y_pred_all[split_idx:] == 1).sum()}")

# Timeline plots
overlay_metrics = [
    ('used_gb',              'Disk Used (GB)',         '#2C3E50'),
    ('write_bytes_sec',      'Write Bytes/sec',        '#8E44AD'),
    ('write_iops',           'Write IOPS',             '#E67E22'),
    ('util_pct',             'Disk Utilization (%)',    '#E74C3C'),
    ('avg_write_latency_ms', 'Avg Write Latency (ms)', '#16A085'),
]

fig, axes = plt.subplots(len(overlay_metrics), 1, figsize=(20, len(overlay_metrics) * 3.5), sharex=True)
fig.suptitle('Anomaly Detection Timeline — Tuned XGBoost', fontsize=16, y=1.01)

for idx, (col, title, color) in enumerate(overlay_metrics):
    ax = axes[idx]
    raw_vals = df_plot_raw[col].values

    # Background line
    ax.plot(timestamps, raw_vals, color=color, linewidth=0.4, alpha=0.6, label=title)

    # Anomaly points
    ax.scatter(timestamps[anom_mask], raw_vals[anom_mask],
               color='red', s=8, zorder=5, alpha=0.7, label=f'Anomalies ({n_anom})')

    # Train/test split line
    ax.axvline(x=timestamps.iloc[split_idx], color='black', linestyle='--',
               linewidth=1.5, alpha=0.7, label='Train/Test Split')

    ax.set_ylabel(col, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    if idx == 0:
        ax.legend(fontsize=8, loc='upper left')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
axes[-1].tick_params(axis='x', rotation=30)
axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

# Anomaly probability heatmap
fig, ax = plt.subplots(figsize=(20, 2))
ax.scatter(timestamps, [1]*len(timestamps), c=y_prob_all, cmap='RdYlGn_r',
           s=2, alpha=0.8, vmin=0, vmax=1)
ax.axvline(x=timestamps.iloc[split_idx], color='black', linestyle='--', linewidth=1.5)
ax.set_yticks([])
ax.set_title('Anomaly Probability Over Time (red = high probability)', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.colorbar(ax.collections[0], ax=ax, label='P(anomaly)', shrink=0.8)
plt.tight_layout()
plt.show()

## 7b. Anomaly Boundary Analysis
Visual comparison of normal vs anomalous data points to identify where the model draws the line.

In [ ]:
# -- Anomaly Boundary Visualization --
# Sort all data points by anomaly probability to see the exact transition zone

sorted_idx = np.argsort(y_prob_all)
sorted_probs = y_prob_all[sorted_idx]

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# --- Left: Full sorted probability curve ---
ax = axes[0]
colors_arr = np.where(sorted_probs >= 0.5, '#E74C3C', '#2ECC71')
ax.scatter(range(len(sorted_probs)), sorted_probs, c=colors_arr, s=1, alpha=0.6)
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
ax.set_xlabel('Data Points (sorted by anomaly probability)', fontsize=11)
ax.set_ylabel('Anomaly Probability', fontsize=11)
ax.set_title('Sorted Anomaly Probability Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Annotate counts
n_normal = (sorted_probs < 0.5).sum()
n_anom_total = (sorted_probs >= 0.5).sum()
ax.text(n_normal * 0.5, 0.15, f'Normal\n{n_normal} pts', ha='center', fontsize=12,
        color='#27AE60', fontweight='bold', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.text(n_normal + n_anom_total * 0.5, 0.85, f'Anomaly\n{n_anom_total} pts', ha='center', fontsize=12,
        color='#E74C3C', fontweight='bold', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# --- Right: Zoomed into the transition zone ---
ax = axes[1]
# Find the crossover point
crossover = np.searchsorted(sorted_probs, 0.5)
zoom_start = max(0, crossover - 300)
zoom_end = min(len(sorted_probs), crossover + 300)
zoom_probs = sorted_probs[zoom_start:zoom_end]
zoom_colors = np.where(zoom_probs >= 0.5, '#E74C3C', '#2ECC71')

ax.scatter(range(len(zoom_probs)), zoom_probs, c=zoom_colors, s=10, alpha=0.7)
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
ax.set_xlabel('Data Points (zoomed around boundary)', fontsize=11)
ax.set_ylabel('Anomaly Probability', fontsize=11)
ax.set_title('Transition Zone (Zoomed)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print boundary statistics
near_boundary = (y_prob_all >= 0.4) & (y_prob_all <= 0.6)
print(f"Points in the 'gray zone' (probability 0.4 - 0.6): {near_boundary.sum()}")
print(f"Points clearly normal (probability < 0.2):         {(y_prob_all < 0.2).sum()}")
print(f"Points clearly anomalous (probability > 0.8):      {(y_prob_all > 0.8).sum()}")


In [ ]:
# -- Normal vs Anomaly: Feature Distribution Comparison --
# Side-by-side histograms showing where normal and anomalous data separate

compare_metrics = [
    ('avg_write_latency_ms', 'Avg Write Latency (ms)'),
    ('write_bytes_sec',      'Write Bytes/sec'),
    ('write_iops',           'Write IOPS'),
    ('read_bytes_sec',       'Read Bytes/sec'),
    ('util_pct',             'Disk Utilization (%)'),
    ('avg_read_latency_ms',  'Avg Read Latency (ms)'),
]

normal_mask = y_pred_all == 0
anomaly_mask = y_pred_all == 1

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.suptitle('Feature Distributions: Normal vs Anomaly', fontsize=16, fontweight='bold', y=1.02)

for idx, (col, title) in enumerate(compare_metrics):
    ax = axes[idx // 3][idx % 3]
    raw_vals = df_plot_raw[col].values

    normal_vals = raw_vals[normal_mask]
    anomaly_vals = raw_vals[anomaly_mask]

    # Use shared bins for fair comparison
    all_vals = raw_vals[np.isfinite(raw_vals)]
    bins = np.linspace(np.percentile(all_vals, 1), np.percentile(all_vals, 99), 80)

    ax.hist(normal_vals, bins=bins, alpha=0.6, color='#2ECC71', label=f'Normal ({len(normal_vals)})',
            density=True, edgecolor='white', linewidth=0.3)
    ax.hist(anomaly_vals, bins=bins, alpha=0.7, color='#E74C3C', label=f'Anomaly ({len(anomaly_vals)})',
            density=True, edgecolor='white', linewidth=0.3)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

# --- Summary statistics table ---
print("\n" + "=" * 85)
print("NORMAL vs ANOMALY — KEY METRIC STATISTICS")
print("=" * 85)
print(f"{'Metric':<25s} | {'Normal (Median)':<18s} | {'Anomaly (Median)':<18s} | {'Anomaly (95th %ile)':<18s}")
print("-" * 85)

for col, title in compare_metrics:
    raw_vals = df_plot_raw[col].values
    norm_med = np.median(raw_vals[normal_mask])
    anom_med = np.median(raw_vals[anomaly_mask])
    anom_95 = np.percentile(raw_vals[anomaly_mask], 95)
    print(f"{title:<25s} | {norm_med:<18.2f} | {anom_med:<18.2f} | {anom_95:<18.2f}")
print("=" * 85)


In [ ]:
# -- Box Plot Comparison: Normal vs Anomaly --
# Compact view showing the spread and outliers for each class

box_metrics = [
    ('avg_write_latency_ms', 'Write Latency (ms)'),
    ('avg_read_latency_ms',  'Read Latency (ms)'),
    ('util_pct',             'Disk Util (%)'),
    ('write_iops',           'Write IOPS'),
]

fig, axes = plt.subplots(1, len(box_metrics), figsize=(22, 6))
fig.suptitle('Normal vs Anomaly — Box Plots', fontsize=16, fontweight='bold', y=1.02)

for idx, (col, title) in enumerate(box_metrics):
    ax = axes[idx]
    raw_vals = df_plot_raw[col].values
    data = [raw_vals[normal_mask], raw_vals[anomaly_mask]]

    bp = ax.boxplot(data, labels=['Normal', 'Anomaly'], patch_artist=True,
                    showfliers=True, flierprops=dict(marker='.', markersize=2, alpha=0.3))
    bp['boxes'][0].set_facecolor('#2ECC71')
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('#E74C3C')
    bp['boxes'][1].set_alpha(0.6)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(alpha=0.2, axis='y')

plt.tight_layout()
plt.show()


## 8. Feature Importance & SHAP Analysis

In [ ]:
# ── XGBoost Feature Importance (Gain) ──

importance_dict = model_tuned.get_booster().get_score(importance_type='gain')

imp_items = []
for fname, score in importance_dict.items():
    # XGBoost names features as f0, f1, ...
    fidx = int(fname[1:])
    if fidx < len(xgb_feature_cols):
        imp_items.append((xgb_feature_cols[fidx], score))

imp_df = pd.DataFrame(imp_items, columns=['feature', 'gain']).sort_values('gain', ascending=False)

print("Feature Importance (Gain):")
print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E74C3C' if 'if_' in f else '#3498DB' for f in imp_df['feature']]
ax.barh(imp_df['feature'], imp_df['gain'], color=colors, alpha=0.8)
ax.invert_yaxis()
ax.set_xlabel('Gain')
ax.set_title('XGBoost Feature Importance (Gain)\n(Red = IF scores, Blue = Engineered features)',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Analysis ──

print("Computing SHAP values (this may take a minute)...")

try:
    # Subsample for SHAP (large datasets are slow)
    SHAP_SAMPLE = min(1000, len(X_test))
    rng = np.random.RandomState(RANDOM_STATE)
    shap_idx = rng.choice(len(X_test), SHAP_SAMPLE, replace=False)
    shap_idx.sort()

    X_shap = pd.DataFrame(X_test[shap_idx], columns=xgb_feature_cols)

    explainer = shap.TreeExplainer(model_tuned)
    shap_values = explainer.shap_values(X_shap)

    # Handle list return (older SHAP: [class_0, class_1])
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Anomaly class

    # 1. Beeswarm (dot) plot — shows feature impact direction
    print("\n--- SHAP Summary Plot (Beeswarm) ---")
    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_values, X_shap, plot_type='dot',
                      max_display=20, show=False)
    plt.title('SHAP Feature Impact (Anomaly Class)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 2. Bar plot — shows mean absolute impact
    print("\n--- SHAP Mean Absolute Impact ---")
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap, plot_type='bar',
                      max_display=20, show=False)
    plt.title('SHAP Mean |Impact| on Anomaly Prediction', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 3. Single anomaly waterfall (if anomalies exist)
    anomaly_shap_idx = np.where(y_pred_tuned[shap_idx] == 1)[0]
    if len(anomaly_shap_idx) > 0:
        print("\n--- SHAP Waterfall for a Single Anomaly ---")
        sample_i = anomaly_shap_idx[0]
        ev = explainer.expected_value
        if isinstance(ev, (list, np.ndarray)):
            ev = ev[1]
        explanation = shap.Explanation(
            values=shap_values[sample_i],
            base_values=ev,
            data=X_shap.iloc[sample_i].values,
            feature_names=xgb_feature_cols,
        )
        fig, ax = plt.subplots(figsize=(10, 8))
        shap.plots.waterfall(explanation, show=False)
        plt.title('SHAP Waterfall — Single Anomaly Example', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print("No anomalies in SHAP sample for waterfall plot.")

    print("\nSHAP analysis complete.")

except Exception as e:
    print(f"SHAP analysis failed (non-critical): {e}")
    print("The model is still valid. SHAP may require a different version.")

## 9. Service-Specific Anomaly Results

Breakdown of detected anomalies by service: **Kafka**, **OpenSearch**, **Logstash**.

For each service, we:
1. Overlay anomalies on the service's disk usage timeline
2. Analyze growth rate anomalies specific to that service
3. Compute correlation between anomalies and service activity

In [ ]:
# ── Service-Specific Anomaly Analysis ──

# Compute per-service growth rates from the aligned raw data
svc_growth = pd.DataFrame()
svc_growth['kafka_growth']     = df_plot_raw['kafka_size_gb'].diff().fillna(0)
svc_growth['opensearch_growth']= df_plot_raw['opensearch_size_gb'].diff().fillna(0)
svc_growth['logstash_growth']  = df_plot_raw['logstash_size_gb'].diff().fillna(0)

service_configs = [
    {
        'name':       'Kafka',
        'size_col':   'kafka_size_gb',
        'growth_col': 'kafka_growth',
        'color':      '#E74C3C',
    },
    {
        'name':       'OpenSearch',
        'size_col':   'opensearch_size_gb',
        'growth_col': 'opensearch_growth',
        'color':      '#3498DB',
    },
    {
        'name':       'Logstash',
        'size_col':   'logstash_size_gb',
        'growth_col': 'logstash_growth',
        'color':      '#2ECC71',
    },
]

fig, axes = plt.subplots(len(service_configs), 2, figsize=(20, len(service_configs) * 4))
fig.suptitle('Service-Specific Anomaly Analysis', fontsize=16, y=1.01)

print("=" * 70)
print("SERVICE-SPECIFIC ANOMALY BREAKDOWN")
print("=" * 70)

for idx, svc in enumerate(service_configs):
    name      = svc['name']
    size_vals = df_plot_raw[svc['size_col']].values
    growth_vals = svc_growth[svc['growth_col']].values
    color     = svc['color']

    # Check if service has meaningful data
    has_data = np.max(size_vals) > 0.001

    # Left: Disk usage timeline with anomalies
    ax_left = axes[idx, 0]
    ax_left.plot(timestamps, size_vals, color=color, linewidth=0.6, alpha=0.7, label=f'{name} Size')
    if has_data:
        ax_left.scatter(timestamps[anom_mask], size_vals[anom_mask],
                        color='red', s=10, zorder=5, alpha=0.7, label='Anomalies')
    ax_left.axvline(x=timestamps.iloc[split_idx], color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax_left.set_title(f'{name} — Disk Usage with Anomalies', fontweight='bold')
    ax_left.set_ylabel('GB')
    ax_left.legend(fontsize=8)
    if not has_data:
        ax_left.text(0.5, 0.5, f'No {name} activity detected',
                     transform=ax_left.transAxes, ha='center', va='center',
                     fontsize=14, color='gray', style='italic')

    # Right: Growth rate with anomalies
    ax_right = axes[idx, 1]
    ax_right.plot(timestamps, growth_vals, color=color, linewidth=0.4, alpha=0.6, label=f'{name} Growth Rate')
    if has_data:
        ax_right.scatter(timestamps[anom_mask], growth_vals[anom_mask],
                         color='red', s=10, zorder=5, alpha=0.7, label='Anomalies')
    ax_right.axvline(x=timestamps.iloc[split_idx], color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax_right.set_title(f'{name} — Growth Rate with Anomalies', fontweight='bold')
    ax_right.set_ylabel('GB/interval')
    ax_right.legend(fontsize=8)
    if not has_data:
        ax_right.text(0.5, 0.5, f'No {name} activity detected',
                      transform=ax_right.transAxes, ha='center', va='center',
                      fontsize=14, color='gray', style='italic')

    # Statistics
    print(f"\n{name}:")
    if has_data:
        # How many anomalies coincide with service growth spikes?
        growth_p95 = np.percentile(growth_vals[growth_vals > 0], 95) if (growth_vals > 0).sum() > 10 else 0
        spike_mask = growth_vals > growth_p95
        overlap = (anom_mask & spike_mask).sum()
        print(f"  Size range:        {np.min(size_vals):.4f} -> {np.max(size_vals):.4f} GB")
        print(f"  Total anomalies:   {n_anom}")
        print(f"  Growth spikes:     {spike_mask.sum()} (above p95 of positive growth)")
        print(f"  Anomalies at spikes: {overlap} ({overlap/max(n_anom,1)*100:.1f}% of anomalies)")

        # Anomaly rate during high vs low service activity
        median_size = np.median(size_vals)
        high_act = size_vals > median_size
        anom_rate_high = anom_mask[high_act].mean() * 100 if high_act.sum() > 0 else 0
        anom_rate_low  = anom_mask[~high_act].mean() * 100 if (~high_act).sum() > 0 else 0
        print(f"  Anomaly rate (high activity): {anom_rate_high:.1f}%")
        print(f"  Anomaly rate (low activity):  {anom_rate_low:.1f}%")
    else:
        print(f"  >> No {name} activity in dataset (all zeros). No service-specific anomalies.")

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
        ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Service vs Anomaly Correlation Analysis ──

print("\n" + "=" * 65)
print("CORRELATION: Service Metrics vs Anomaly Probability")
print("=" * 65)

corr_data = pd.DataFrame({
    'anomaly_prob': y_prob_all,
    'kafka_size': df_plot_raw['kafka_size_gb'].values,
    'kafka_growth': svc_growth['kafka_growth'].values,
    'opensearch_size': df_plot_raw['opensearch_size_gb'].values,
    'opensearch_growth': svc_growth['opensearch_growth'].values,
    'write_bytes': df_plot_raw['write_bytes_sec'].values,
    'write_iops': df_plot_raw['write_iops'].values,
    'util_pct': df_plot_raw['util_pct'].values,
})

corr_with_anom = corr_data.corr()['anomaly_prob'].drop('anomaly_prob').sort_values(ascending=False)
print("\nCorrelation with anomaly probability:")
for feat, r in corr_with_anom.items():
    bar = '#' * int(abs(r) * 30)
    print(f"  {feat:25s}: {r:+.4f}  {bar}")

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ['#E74C3C' if v > 0 else '#3498DB' for v in corr_with_anom.values]
ax.barh(corr_with_anom.index, corr_with_anom.values, color=colors_bar, alpha=0.8)
ax.set_xlabel('Pearson Correlation with P(anomaly)')
ax.set_title('Service Metrics vs Anomaly Probability', fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 10. Save Artifacts

In [ ]:
import joblib
import json

os.makedirs('artifacts_v8', exist_ok=True)

# Save IF models
for group_name, model in if_models.items():
    joblib.dump(model, f'artifacts_v8/if_{group_name}_tuned.pkl')

# Save scalers
for group_name, scaler in scalers.items():
    joblib.dump(scaler, f'artifacts_v8/scaler_{group_name}.pkl')

# Save XGBoost models
model_tuned.save_model('artifacts_v8/xgb_tuned.json')
model_default.save_model('artifacts_v8/xgb_default.json')

# Save Optuna study
joblib.dump(xgb_study, 'artifacts_v8/optuna_xgb_study.pkl')
for group_name, study in if_studies.items():
    joblib.dump(study, f'artifacts_v8/optuna_if_{group_name}_study.pkl')

# Save feature configuration
config = {
    'xgb_feature_cols': xgb_feature_cols,
    'if_feature_groups': if_feature_groups,
    'if_best_params': if_best_params,
    'xgb_best_params': xgb_study.best_params,
    'rolling_window': WINDOW,
    'drop_rows': DROP_ROWS,
    'pseudo_contamination': PSEUDO_CONTAMINATION,
    'random_state': RANDOM_STATE,
}
with open('artifacts_v8/feature_config.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)

# Save scored dataset
output_df = df_plot_raw.copy()
output_df['anomaly_prob'] = y_prob_all
output_df['anomaly_pred'] = y_pred_all
output_df['if_throughput_score'] = df_feat['if_throughput_score'].values
output_df['if_latency_score']    = df_feat['if_latency_score'].values
output_df['if_capacity_score']   = df_feat['if_capacity_score'].values
output_df.to_csv('artifacts_v8/pipeline_metrics_scored.csv', index=False)

print("Artifacts saved to artifacts_v8/:")
for f in sorted(os.listdir('artifacts_v8')):
    size = os.path.getsize(f'artifacts_v8/{f}')
    print(f"  {f:45s} {size/1024:>8.1f} KB")
# ── Zip and Download Artifacts (Google Colab) ──
import shutil
print("\nZipping artifacts...")
shutil.make_archive('artifacts_v8', 'zip', 'artifacts_v8')

try:
    from google.colab import files
    print("Triggering automatic download to your local machine...")
    files.download('artifacts_v8.zip')
    print("Download initiated! Please check your browser's downloads.")
except ImportError:
    print("Not running in Google Colab. Artifacts are saved locally as artifacts_v8.zip.")


## Summary

### What was done
1. **EDA** with comprehensive time-series, distribution, correlation, and service-specific visualizations
2. **Trend-aware feature engineering**: rate-of-change, rolling Z-scores, volatility, I/O ratios (21 features)
3. **Three Isolation Forest models** (throughput, latency, capacity) with **Optuna-tuned** hyperparameters
4. **XGBoost classifier** with **100-trial Bayesian hyperparameter optimization** using TimeSeriesSplit CV
5. **Temporal train/test split** (no data leakage) instead of random splitting
6. **SHAP explainability** for feature impact analysis
7. **Service-specific analysis** for Kafka, OpenSearch, and Logstash
8. **Default vs Tuned comparison** quantifying the value of hyperparameter tuning

### Key Caveats
- All evaluation is against **pseudo-labels** (no ground-truth available)
- The Isolation Forest models were fit on the full dataset (unsupervised, acceptable practice)
- The temporal split ensures XGBoost generalizes to future patterns, not just memorizes training data
- Visual inspection of the anomaly timeline is the most reliable validation in this unsupervised setting